# LeRobot: End-to-End Policy Training, Export & Inference

This notebook demonstrates the standalone LeRobot workflow for policy training, export, and inference. This process allows for the creation of lightweight, portable policy packages that can be used for robot control without requiring a full training environment.

### Workflow Overview
1. **Train a Policy**: In this example, we instantiate a policy with random weights to simulate a trained model.
2. **Export to policy_package**: Convert the PyTorch policy to a standardized format (ONNX or OpenVINO) for cross-platform inference.
3. **Standalone Inference**: Load the exported package and run predictions on simulated observations.

### Prerequisites
- `lerobot` library installed
- `torch` for policy instantiation and export
- `numpy` for inference inputs and outputs
- Optional: `openvino` for the OpenVINO backend

## 1. Train an ACT Policy (Simulated)

While production models are trained using `lerobot.scripts.train` or the `Trainer` class, we can simulate a trained policy by instantiating an `ACTPolicy` with random weights. This allows us to test the export and inference pipeline without waiting for training to complete.

In [ ]:
import torch
from lerobot.configs.types import FeatureType, NormalizationMode, PolicyFeature
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.utils.constants import ACTION, OBS_STATE

input_features = {
    OBS_STATE: PolicyFeature(type=FeatureType.STATE, shape=(6,)),
    "observation.images.top": PolicyFeature(type=FeatureType.VISUAL, shape=(3, 84, 84)),
}
output_features = {
    ACTION: PolicyFeature(type=FeatureType.ACTION, shape=(6,)),
}

config = ACTConfig(
    device="cpu",
    chunk_size=10,
    n_action_steps=10,
    dim_model=64,
    n_heads=2,
    dim_feedforward=128,
    n_encoder_layers=2,
    n_decoder_layers=2,
    n_vae_encoder_layers=2,
    use_vae=False,
    latent_dim=16,
    vision_backbone="resnet18",
    pretrained_backbone_weights=None,
    input_features=input_features,
    output_features=output_features,
    normalization_mapping={
        "VISUAL": NormalizationMode.MEAN_STD,
        "STATE": NormalizationMode.MEAN_STD,
        "ACTION": NormalizationMode.MEAN_STD,
    },
)

policy = ACTPolicy(config)
policy.eval()
print(f"Policy: {policy.__class__.__name__}")
print(f"Parameters: {sum(p.numel() for p in policy.parameters()):,}")

## 2. Export the Policy

The `export_policy` utility packages the model weights, configuration, and any necessary artifacts into a single directory. We specify the `onnx` backend to create a portable model file.

In [ ]:
from lerobot.export import export_policy
import os
import json
from pathlib import Path

batch = {
    "observation.state": torch.randn(1, 6),
    "observation.images.top": torch.randn(1, 3, 84, 84),
}

package_path = export_policy(
    policy,
    "./exported_act",
    backend="onnx",
    example_batch=batch,
    include_normalization=False,
)
print(f"Exported to: {package_path}")

# Show package contents
print("\nPackage contents:")
for f in os.listdir(package_path):
    print(f" - {f}")

# Show manifest content
manifest_path = Path(package_path) / "manifest.json"
with open(manifest_path, "r") as f:
    manifest = json.load(f)
    print("\nManifest content:")
    print(json.dumps(manifest, indent=2))

## 3. Load and Run Inference (LeRobot Standalone)

The exported policy can be loaded into an `InferenceRunner`. This runner handles model loading and provides a simple API for predicting actions. Note that the runner accepts NumPy arrays as input.

In [ ]:
from lerobot.export import load_exported_policy
import numpy as np

runner = load_exported_policy(package_path, backend="onnx", device="cpu")
print(f"Runner type: {type(runner).__name__}")

observation = {
    "observation.state": np.random.randn(1, 6).astype(np.float32),
    "observation.images.top": np.random.randn(1, 3, 84, 84).astype(np.float32),
}

action_chunk = runner.predict_action_chunk(observation)
print(f"Action chunk shape: {action_chunk.shape}")
print(f"Action chunk dtype: {action_chunk.dtype}")
print(f"Action range: [{action_chunk.min():.4f}, {action_chunk.max():.4f}]")

## 4. Action Chunking (Single-Action Control Loop)

For real-time control, policies often output a chunk of actions (e.g., the next 10 steps). The `InferenceRunner` includes internal logic to queue these actions. Calling `predict_action` retrieves the next action from the current chunk or generates a new chunk if needed.

In [ ]:
for step in range(20):
    action = runner.predict_action(observation)
    print(f"Step {step}: action shape={action.shape}, values={action[:3]}")
    
    # In real robot control:
    # env.step(action)
    # observation = env.get_observation()

## 5. OpenVINO Backend (Optional)

LeRobot also supports the OpenVINO backend, which can provide significant performance improvements on CPU. We can load the same exported directory using the OpenVINO runner.

In [ ]:
try:
    ov_runner = load_exported_policy(package_path, backend="openvino", device="cpu")
    ov_action = ov_runner.predict_action_chunk(observation)
    
    # Compare numerical parity
    np.testing.assert_allclose(action_chunk, ov_action, atol=1e-5)
    print("✅ ONNX and OpenVINO outputs match!")
except ImportError:
    print("OpenVINO not installed - skipping")

## 6. Supported Policies

LeRobot supports a variety of policies for export. Depending on the architecture, the export process may involve different phases.

| Policy | Type | Backend Support |
| :--- | :--- | :--- |
| ACT | single_pass | ONNX, OpenVINO |
| Diffusion | iterative | ONNX, OpenVINO |
| PI0 | two_phase | ONNX, OpenVINO |
| PI05 | two_phase | ONNX, OpenVINO |
| SmolVLA | two_phase | ONNX, OpenVINO |

Note: Large models like PI0, PI05, and SmolVLA require CUDA for the export process due to model size and memory requirements.

## 7. Cleanup

Remove the exported directory created during this demonstration.

In [ ]:
import shutil
if os.path.exists(package_path):
    shutil.rmtree(package_path)
    print(f"Removed {package_path}")